# Phase 05 · Bài 04 — GloVe, FastText và Subword

**Mục tiêu:** phân biệt GloVe, FastText và BPE theo tín hiệu học/đầu ra; hiểu subword giúp gì cho từ hiếm và OOV.

## 1. GloVe: học từ thống kê đồng xuất hiện toàn cục

GloVe tạo ma trận X, trong đó X[i,j] là số lần từ j xuất hiện trong cửa sổ ngữ cảnh của từ i. Có thể giảm đóng góp của ngữ cảnh xa bằng trọng số nghịch đảo khoảng cách. Mục tiêu là làm tích vô hướng vector cùng bias khớp log của số đồng xuất hiện:

w_i · w̃_j + b_i + b̃_j ≈ log(X[i,j])

Loss bình phương được nhân trọng số f(X[i,j]) để các cặp cực hiếm hoặc cực phổ biến không chi phối quá mức. Một hàm thường dùng tăng theo count ở vùng thấp rồi bão hòa ở ngưỡng x_max. Sau huấn luyện, người ta thường cộng vector center và context để có embedding cuối.

Khác biệt trực giác: Skip-gram tạo các cặp cục bộ rồi tối ưu dự đoán; GloVe bắt đầu từ thống kê đồng xuất hiện tổng hợp của toàn corpus rồi factorize theo objective có trọng số. Cả hai đều tạo vector tĩnh theo từ.

## 2. FastText: cấu tạo từ các mảnh ký tự

FastText bọc từ bằng ranh giới như <where>, tạo character n-gram ở nhiều độ dài rồi biểu diễn từ bằng tổng hoặc tổ hợp vector n-gram cùng vector từ. Nếu một từ mới chia sẻ các n-gram đã học, mô hình vẫn có thể tạo vector cho nó. Điều này hữu ích với biến tố, từ hiếm, lỗi chính tả gần nhau và từ mới có hình thái quen thuộc.

N-gram không đảm bảo hiểu nghĩa. Hai chuỗi giống mặt chữ có thể khác nghĩa; từ ngắn có thể có ít mảnh hữu ích; n-gram chưa từng học không mang vector có ích. FastText giảm vấn đề OOV nhưng không loại bỏ nhu cầu dữ liệu, tokenizer và đánh giá phù hợp.

## 3. BPE: học bộ phận token thay vì vector từ

Byte-Pair Encoding bắt đầu từ ký tự hoặc byte. Ở mỗi vòng, đếm tần suất các cặp token liền kề trong corpus (có tính tần suất từ), gộp cặp phổ biến nhất thành một token mới và ghi merge vào danh sách. Lặp đến số merge/vocabulary mong muốn. Khi tokenize văn bản mới, áp dụng merges theo đúng thứ tự đã học.

BPE tạo chuỗi token/subword ID cho model, không tự tạo embedding ngữ nghĩa như GloVe/FastText. Model sau đó học embedding cho từng token BPE trong quá trình huấn luyện. Ranh giới từ như </w> trong demo giúp phân biệt một chuỗi ở cuối từ. Byte-level BPE có thể bao phủ chuỗi rất lạ vì cuối cùng nội dung vẫn được tạo từ byte.

Tradeoff chính là kích thước vocabulary so với độ dài chuỗi: merge nhiều thì từ quen thuộc gọn hơn nhưng vocabulary lớn hơn; merge ít thì vocabulary nhỏ nhưng chuỗi dài, tốn vị trí context hơn. Từ rất hiếm có thể tách thành nhiều mảnh. Tokenizer đã dùng để train là một phần của model; đổi tokenizer lúc inference làm ID không còn ánh xạ đúng embedding.

## 4. Chọn đúng loại biểu diễn

| Phương pháp | Đầu vào/tín hiệu | Đầu ra | Điểm mạnh | Giới hạn chính |
|---|---|---|---|---|
| GloVe | Ma trận đồng xuất hiện | Vector tĩnh mỗi từ | Tận dụng thống kê toàn corpus | Từ OOV cần fallback |
| FastText | Từ và character n-gram | Vector tĩnh tổng hợp từ mảnh | Có thể ghép vector cho từ OOV | Mảnh ký tự không bảo đảm ý nghĩa |
| BPE | Tần suất cặp token/byte | Chuỗi subword token ID | Bao phủ văn bản chưa thấy | Segmentation dài hoặc vocabulary lớn |
| Word2Vec | Cặp center/context cục bộ | Vector tĩnh mỗi từ | Học trực tiếp từ ngữ cảnh lân cận | OOV và đa nghĩa |

Notebook chạy code mẫu có n-gram FastText và BPE. Phần thực hành dùng vector đếm n-gram chỉ để quan sát mức độ trùng mảnh, rồi tự học/apply merges BPE trên corpus nhỏ. Các kết quả đó minh họa thuật toán chứ không phải checkpoint production.

## A. Chạy code mẫu của bài

Đầu ra cho biết n-gram chung của từ đã biết/từ mở rộng và segmentation BPE.

In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "04-glove-fasttext-subword"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "GloVe, FastText, and Subword Embeddings")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: GloVe, FastText, and Subword Embeddings
Lệnh: phases\05-nlp-foundations-to-advanced\04-glove-fasttext-subword\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
=== FastText n-grams ===
where        15 grams, e.g., ['<wh', '<whe', '<wher', '<where', '<where>']
whereupon    31 grams, e.g., ['<wh', '<whe', '<wher', '<where', '<whereupon>']
shared n-grams between where / whereupon: 10

=== BPE on toy corpus ===
learned 10 merges:
  'w' + 'e' -> 'we'
  's' + 't' -> 'st'
  'st' + '</w>' -> 'st</w>'
  'l' + 'o' -> 'lo'
  'we' + 'st</w>' -> 'west</w>'
  'n' + 'e' -> 'ne'
  'ne' + 'west</w>' -> 'newest</w>'
  'lo' + 'w' -> 'low'
  'low' + '</w>' -> 'low</w>'
  'we' + 'r' -> 'wer'

lowest     -> ['lo', 'west</w>']
slowest    -> ['s', 'lo', 'west</w>']
lower      -> ['lo', 'wer', '</w>']
newish     -> ['ne', 'w', 'i', 's', 'h', '</w>']


## B. Thực hành riêng — character n-gram và từ OOV

Để phép so sánh dễ đọc, ta biến mỗi từ thành vector đếm n-gram rồi so bằng cosine. Đây là bag-of-substrings đơn giản, **không phải FastText đã huấn luyện**; ta quan sát việc từ chưa từng thấy vẫn chia sẻ mảnh ký tự.

In [2]:
import math
from collections import Counter

def char_ngrams(word, n_min=3, n_max=5):
    wrapped = f"<{word.lower()}>"
    return {wrapped[i:i+n] for n in range(n_min, n_max+1)
            for i in range(len(wrapped)-n+1)}

words = ["where", "whereupon", "whenever", "dog"]
all_grams = sorted(set().union(*(char_ngrams(word) for word in words)))
def ngram_vector(word):
    counts = Counter(char_ngrams(word))
    return [counts[gram] for gram in all_grams]

def cosine(left, right):
    dot = sum(a*b for a, b in zip(left, right))
    norm_left = math.sqrt(sum(a*a for a in left))
    norm_right = math.sqrt(sum(b*b for b in right))
    return dot / (norm_left * norm_right) if norm_left and norm_right else 0.0

for left, right in (("where", "whereupon"), ("where", "whenever"), ("where", "dog")):
    shared = char_ngrams(left) & char_ngrams(right)
    print(f"{left:10s} vs {right:10s}: shared={len(shared):2d}, cosine={cosine(ngram_vector(left), ngram_vector(right)):.3f}")
print("\nTừ OOV vẫn chia sẻ mảnh ký tự với những từ đã biết trong ví dụ này.")

where      vs whereupon : shared= 9, cosine=0.530
where      vs whenever  : shared= 3, cosine=0.189
where      vs dog       : shared= 0, cosine=0.000

Từ OOV vẫn chia sẻ mảnh ký tự với những từ đã biết trong ví dụ này.


## C. Thực hành riêng — học và áp dụng BPE

BPE đếm cặp theo tần suất của từng từ trong corpus. Mỗi vòng gộp một cặp liền kề; khi áp dụng lên từ mới, chạy đúng thứ tự merge đã học.

In [3]:
def learn_bpe(corpus, merge_count):
    vocab = {tuple(list(word) + ["</w>"]): frequency
             for word, frequency in corpus.items()}
    merges = []
    for _ in range(merge_count):
        pair_counts = Counter()
        for symbols, frequency in vocab.items():
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += frequency
        if not pair_counts:
            break
        best_pair, best_count = pair_counts.most_common(1)[0]
        merges.append((best_pair, best_count))
        updated = {}
        for symbols, frequency in vocab.items():
            result, i = [], 0
            while i < len(symbols):
                if i + 1 < len(symbols) and symbols[i:i+2] == best_pair:
                    result.append(symbols[i] + symbols[i+1])
                    i += 2
                else:
                    result.append(symbols[i])
                    i += 1
            result = tuple(result)
            updated[result] = updated.get(result, 0) + frequency
        vocab = updated
    return merges

def apply_bpe(word, merges):
    symbols = list(word) + ["</w>"]
    for (left, right), _frequency in merges:
        result, i = [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and (symbols[i], symbols[i+1]) == (left, right):
                result.append(left + right)
                i += 2
            else:
                result.append(symbols[i])
                i += 1
        symbols = result
    return symbols

corpus = Counter({"low": 5, "lower": 2, "lowest": 4, "newest": 6, "newer": 2, "widest": 3})
merges = learn_bpe(corpus, merge_count=12)
print("learned merges (pair, training frequency):")
for (left, right), count in merges:
    print(f"  {left!r} + {right!r} -> {(left+right)!r}  (count={count})")
print("\nsegmentations:")
for word in ("lowest", "slowest", "lower", "newish"):
    print(f"  {word:8s} -> {apply_bpe(word, merges)}")

learned merges (pair, training frequency):
  'w' + 'e' -> 'we'  (count=14)
  's' + 't' -> 'st'  (count=13)
  'st' + '</w>' -> 'st</w>'  (count=13)
  'l' + 'o' -> 'lo'  (count=11)
  'we' + 'st</w>' -> 'west</w>'  (count=10)
  'n' + 'e' -> 'ne'  (count=8)
  'ne' + 'west</w>' -> 'newest</w>'  (count=6)
  'lo' + 'w' -> 'low'  (count=5)
  'low' + '</w>' -> 'low</w>'  (count=5)
  'we' + 'r' -> 'wer'  (count=4)
  'wer' + '</w>' -> 'wer</w>'  (count=4)
  'lo' + 'west</w>' -> 'lowest</w>'  (count=4)

segmentations:
  lowest   -> ['lowest</w>']
  slowest  -> ['s', 'lowest</w>']
  lower    -> ['lo', 'wer</w>']
  newish   -> ['ne', 'w', 'i', 's', 'h', '</w>']


## D. Tự kiểm tra

1. GloVe dùng tín hiệu nào khác Skip-gram?
2. Vì sao character n-gram giúp FastText biểu diễn từ OOV?
3. Vì sao BPE phải áp dụng merge theo đúng thứ tự đã học?
4. Vì sao không nên thay tokenizer của model đã huấn luyện?

**Giới hạn:** n-gram bag-of-substrings chỉ minh họa mảnh ký tự; corpus BPE quá nhỏ để dùng làm tokenizer sản xuất.